In [4]:
import hashlib
import binascii
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes

# Load the ResNet50 model pre-trained on ImageNet (without top layers, to extract features)
model = ResNet50(weights='imagenet', include_top=False)

# Preprocess the image for ResNet50
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)  # Normalize the image for ResNet50
    return img_array

# Extract features from the image using ResNet50
def extract_features(model, img_array):
    features = model.predict(img_array)
    return features

# Flatten the features into a 1D sequence
def flatten_features(features):
    print(features.flatten())
    return features.flatten()

# Hash the flattened features using SHA-256
def hash_features(features_flat):
    hash_object = hashlib.sha256(features_flat.tobytes())  # Hash the byte sequence of features
    hashed_features = hash_object.hexdigest()
    return hashed_features

# Generate RSA key pairs based on the hashed feature sequence
def generate_rsa_key_pair():
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
        backend=default_backend()
    )
    public_key = private_key.public_key()

    return private_key, public_key

# Serialize keys and save them to files
def save_keys(private_key, public_key):
    private_pem = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

    public_pem = public_key.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    )

    with open("image_private_key.pem", "wb") as f:
        f.write(private_pem)

    with open("image_public_key.pem", "wb") as f:
        f.write(public_pem)

# Load the keys back from files
def load_keys():
    with open("image_private_key.pem", "rb") as f:
        private_key = serialization.load_pem_private_key(
            f.read(),
            password=None,
            backend=default_backend()
        )

    with open("image_public_key.pem", "rb") as f:
        public_key = serialization.load_pem_public_key(
            f.read(),
            backend=default_backend()
        )

    return private_key, public_key

# Encrypt and decrypt a small message to test the keys
def encrypt_decrypt_test(public_key, private_key):
    message = b"Hello, this is a test message."

    # Encrypt using loaded public key
    ciphertext = public_key.encrypt(
        message,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    # Decrypt using loaded private key
    plaintext = private_key.decrypt(
        ciphertext,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    print("Original message:", message.decode())
    print("Decrypted message:", plaintext.decode())

# Main function to tie everything together
def main(img_path):
    # Preprocess and extract features from the image
    img_array = preprocess_image(img_path)
    features = extract_features(model, img_array)

    # Flatten and hash the features
    features_flat = flatten_features(features)
    hashed_features = hash_features(features_flat)

    # Convert hash to seed (optional: could use this for deterministic key generation)
    seed = int(hashed_features, 16)  # Convert the hex hash to an integer
    print("Hashed Features (SHA-256):", hashed_features)

    # Generate RSA key pairs
    private_key, public_key = generate_rsa_key_pair()

    # Save and load the keys to/from files
    save_keys(private_key, public_key)
    loaded_private_key, loaded_public_key = load_keys()

    # Test encryption/decryption
    encrypt_decrypt_test(loaded_public_key, loaded_private_key)

# Run the main function on an example image
img_path = 'Designer (1).jpeg'
main(img_path)


[0. 0. 0. ... 0. 0. 0.]
Hashed Features (SHA-256): ce46067ef92d92b24f60a6bee12c86d1faa74c8680a60f1e54542e432a5f855a
Original message: Hello, this is a test message.
Decrypted message: Hello, this is a test message.
